# 제품 이상여부 판별 프로젝트 (본선)


In [2]:
# Catboost
!pip install catboost --quiet


[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


## 1. 데이터 불러오기


### 필수 라이브러리


In [1]:
import os
from pprint import pprint
import random

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from tqdm import tqdm

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
SEED = 999
seed_everything(SEED)

### 데이터 읽어오기


In [19]:
# Collect Date 컬럼에서 년월일,시분초 분리하는 함수
def split_date(data, col, drop=True):
    df = data.copy()
    
    df[col] = df[col].astype('str')
    
    df['date_'+col] = df[col].apply(lambda x: x.split(' ')[0])
    df['time_'+col] = df[col].apply(lambda x: x.split(' ')[1])

    df['year_'+col] = df['date_'+col].apply(lambda x: x.split('-')[0]).astype(str)
    df['month_'+col] = df['date_'+col].apply(lambda x: x.split('-')[1]).astype(str)
    df['day_'+col] = df['date_'+col].apply(lambda x: x.split('-')[2]).astype(str)
    df['hour_'+col] = df['time_'+col].apply(lambda x: x.split(':')[0]).astype(str)
    df['min_'+col] = df['time_'+col].apply(lambda x: x.split(':')[1]).astype(str)
    
    if drop:
        df.drop([col, 'date_'+col, 'time_'+col], axis=1, inplace=True)
    
    return df

# 두 열 데이터의 시간 차를 파생변수로
def create_diff_column(data, column1, column2):
    df = data.copy()
    
    # datetime 형식으로 변환
    data1 = df[column1]
    data2 = df[column2]
    data1 = pd.to_datetime(data1, errors='coerce')
    data2 = pd.to_datetime(data2, errors='coerce')

    # 결과를 저장할 새로운 열 초기화
    fill_diff = pd.Series(pd.to_timedelta('0 days 00:00:00'), index=data1.index)

    count = 0


    for i in range(len(data1)):
        if data1[i] == data2[i]:
            fill_diff[i] = pd.to_timedelta('0 days 00:00:00')  # 동일한 경우
        else:
            time_difference = data2[i] - data1[i]
            fill_diff[i] = time_difference  # 다를 경우 시간 차이로 채우기

    return fill_diff

# equipment 매치 여부
def add_equipment_match_column(df):
    # 마지막 숫자 추출
    last_char_dam = df["Equipment_Dam"].str[-1]
    last_char_fill1 = df["Equipment_Fill1"].str[-1]
    last_char_fill2 = df["Equipment_Fill2"].str[-1]

    # 파생변수 생성 (모두 일치하면 1, 그렇지 않으면 0)
    df['equipment_match'] = ((last_char_dam == last_char_fill1) & 
                             (last_char_fill1 == last_char_fill2)).astype(int)
    return df

In [3]:
def process_and_merge_data(train_data, test_data, hand_data):

    # hand_data 처리 (날짜 및 시간을 분리)
    df = hand_data.copy()
    df = df.rename(columns={'day': 'date'})
    
    # date 컬럼의 값을 문자열로 변환 후 6자리까지만 추출 (YYMMDD)
    df['date'] = df['date'].astype(str).apply(lambda x: x[:6])
    
    # 연도, 월, 일로 나누기 (앞에서 2자리: 연도, 중간 2자리: 월, 마지막 2자리: 일)
    df['year'] = df['date'].apply(lambda x: int('20' + x[:2]))  # 20XX로 변환
    df['month'] = df['date'].apply(lambda x: int(x[2:4]))
    df['day'] = df['date'].apply(lambda x: int(x[4:6]))
    
    # 'YYYY-MM-DD' 형식의 date 컬럼 재생성
    df['date'] = df.apply(lambda row: f"{row['year']}-{row['month']:02d}-{row['day']:02d}", axis=1)
    
    # start_time과 end_time을 시, 분, 초로 분리
    df['start_time'] = pd.to_datetime(df['start_time'], format='%H:%M:%S')
    df['end_time'] = pd.to_datetime(df['end_time'], format='%H:%M:%S')

    # start_time의 시, 분, 초 분리
    df['start_hour'] = df['start_time'].dt.hour
    df['start_min'] = df['start_time'].dt.minute
    df['start_sec'] = df['start_time'].dt.second
    
    # end_time의 시, 분, 초 분리
    df['end_hour'] = df['end_time'].dt.hour
    df['end_min'] = df['end_time'].dt.minute
    df['end_sec'] = df['end_time'].dt.second
    
    # 불필요한 컬럼 삭제 (기존 date, start_time, end_time 형식은 필요 시 유지 가능)
    df.drop(columns=['start_time', 'end_time'], inplace=True)

    # train_data에서 'Collect Date_Dam'을 datetime으로 변환
    train_data['Collect Date_Dam'] = pd.to_datetime(train_data['Collect Date_Dam'])
    test_data['Collect Date_Dam'] = pd.to_datetime(test_data['Collect Date_Dam'])

    # hand_data에서 각각의 시간 정보를 직접 조합하여 datetime 생성
    df['start_time'] = pd.to_datetime(df[['year', 'month', 'day']].astype(str).agg('-'.join, axis=1)) + \
                       pd.to_timedelta(df['start_hour'], unit='h') + \
                       pd.to_timedelta(df['start_min'], unit='m') + \
                       pd.to_timedelta(df['start_sec'], unit='s')

    df['end_time'] = pd.to_datetime(df[['year', 'month', 'day']].astype(str).agg('-'.join, axis=1)) + \
                     pd.to_timedelta(df['end_hour'], unit='h') + \
                     pd.to_timedelta(df['end_min'], unit='m') + \
                     pd.to_timedelta(df['end_sec'], unit='s')

    def merge_data(input_data):
        """
        내부적으로 train_data 또는 test_data에 대해 병합 작업을 수행하는 함수.
        """
        results = []
        for idx, input_row in input_data.iterrows():
            # train_data나 test_data의 시간이 hand_data의 시간 범위 내에 있는지 확인
            matching_hand_data = df[
                (input_row['Collect Date_Dam'] >= df['start_time']) &
                (input_row['Collect Date_Dam'] <= df['end_time'])
            ]
            if matching_hand_data.empty:
                # 범위 내에 없다면 NaN으로 채움
                empty_hand_data = {col: np.nan for col in df.columns}
                results.append({**input_row.to_dict(), **empty_hand_data})
            else:
                # 조건을 만족하는 hand_data의 행들을 input_data에 병합
                for _, hand_row in matching_hand_data.iterrows():
                    results.append({**input_row.to_dict(), **hand_row.to_dict()})
                    
        # 결과를 DataFrame으로 변환
        merged_data = pd.DataFrame(results)
        merged_data = merged_data.drop(columns=['start_time', 'end_time'])
        return merged_data
    
    # train_data와 test_data 각각에 대해 병합 작업 수행
    final_train_data = merge_data(train_data)
    final_test_data = merge_data(test_data)

    return final_train_data, final_test_data

In [4]:
# 라벨링 매핑을 위한 딕셔너리 정의
Stage1_product_mapping = {
    ('6500', '1'): 'Stage1_1',
    ('6500', '6'): 'Stage1_2',
    ('6500', '9'): 'Stage1_3',
    ('5800', '1'): 'Stage1_4',
    ('5800', '6'): 'Stage1_5',
    ('5800', '17'): 'Stage1_6',
    ('6000', '1'): 'Stage1_7',
    ('6000', '6'): 'Stage1_8',
    ('6200', '1'): 'Stage1_9',
    ('4000', '4000'): 'Stage1_10',
    ('4000', '1'): 'Stage1_11',
    ('4000', '3'): 'Stage1_12',
    ('5000', '5000'): 'Stage1_13',
    ('9000', '9000'): 'Stage1_14',
}

Stage2_product_mapping = {
    ('4000', '4000'): 'Stage2_1',
    ('5000', '5000'): 'Stage2_2',
    ('5000', '4000'): 'Stage2_3',
    ('5300', '5800'): 'Stage2_4',
    ('5500', '6000'): 'Stage2_5',
    ('5500', '6200'): 'Stage2_6',
    ('5500', '6500'): 'Stage2_7',
    ('6000', '6500'): 'Stage2_8',
    ('6500', '6500'): 'Stage2_9',
    ('8000', '4000'): 'Stage2_10',
    ('9000', '9000'): 'Stage2_11',
    ('12000', '6500'): 'Stage2_12',
    ('12000', '12000'): 'Stage2_13'
}

Stage3_product_mapping = {
    ('9000', '9000'): 'Stage3_1',
    ('6500', '5500'): 'Stage3_2',
    ('6500', '5800'): 'Stage3_3',
    ('6500', '6000'): 'Stage3_4',
    ('6500', '6500'): 'Stage3_5',
    ('6500', '12000'): 'Stage3_6',
    ('4000', '4000'): 'Stage3_7',
    ('4000', '5000'): 'Stage3_8',
    ('4000', '8000'): 'Stage3_9',
    ('5000', '5000'): 'Stage3_10',
    ('6200', '5500'): 'Stage3_11',
    ('6000', '5500'): 'Stage3_12',
    ('5800', '5300'): 'Stage3_13'
}

# 라벨링 매핑을 위한 딕셔너리 정의
code_mapping = {
    # stage1_1
    (1, 7, 2): 'code_1',
    (1, 7, 3): 'code_2',
    (1, 9, 5): 'code_3',
    (1, 12, 6): 'code_4',
    # stage1_2
    (2, 7, 2): 'code_5',
    (2, 7, 12): 'code_6',
    (2, 8, 4): 'code_7',
    (2, 9, 5): 'code_8',
    # stage1_3~9
    (3, 7, 2): 'code_9',
    (4, 4, 13): 'code_10',
    (5, 4, 13): 'code_11',
    (6, 4, 13): 'code_12',
    (7, 5, 12): 'code_13',
    (8, 5, 12): 'code_14',
    (9, 6, 11): 'code_15',
    # stage1_10
    (10, 2, 7): 'code_16',
    (10, 11, 7): 'code_17',
    # stage1_11
    (11, 1, 7): 'code_18',
    (11, 3, 8): 'code_19',
    (11, 10, 9): 'code_20',
    # stage1_12~13
    (12, 1, 7): 'code_21',
    (13, 11, 10): 'code_22',
    # stage1_14
    (14, 11, 1): 'code_23',
    (14, 13, 1): 'code_24'
}

# 추출하고자 하는 열 이름 리스트 생성
Circles_lines_drop = [
    'Stage1 Circle1 Distance Speed Collect Result_Dam',
    'Stage1 Circle2 Distance Speed Collect Result_Dam',
    'Stage1 Circle3 Distance Speed Collect Result_Dam',
    'Stage1 Circle4 Distance Speed Collect Result_Dam',
    'Stage2 Circle1 Distance Speed Collect Result_Dam',
    'Stage2 Circle2 Distance Speed Collect Result_Dam',
    'Stage2 Circle3 Distance Speed Collect Result_Dam',
    'Stage2 Circle4 Distance Speed Collect Result_Dam',
    'Stage3 Circle1 Distance Speed Collect Result_Dam',
    'Stage3 Circle2 Distance Speed Collect Result_Dam',
    'Stage3 Circle3 Distance Speed Collect Result_Dam',
    'Stage3 Circle4 Distance Speed Collect Result_Dam',
    'product_code_stage1',
    'product_code_stage2',
    'product_code_stage3',     
    'Stage1 Line1 Distance Speed Collect Result_Dam',
    'Stage1 Line2 Distance Speed Collect Result_Dam',
    'Stage1 Line3 Distance Speed Collect Result_Dam',
    'Stage1 Line4 Distance Speed Collect Result_Dam',
    'Stage2 Line1 Distance Speed Collect Result_Dam',
    'Stage2 Line2 Distance Speed Collect Result_Dam',
    'Stage2 Line3 Distance Speed Collect Result_Dam',
    'Stage2 Line4 Distance Speed Collect Result_Dam',
    'Stage3 Line1 Distance Speed Collect Result_Dam',
    'Stage3 Line2 Distance Speed Collect Result_Dam',
    'Stage3 Line3 Distance Speed Collect Result_Dam',
    'Stage3 Line4 Distance Speed Collect Result_Dam'
    
]

# Stage별 product code 만드는 함수
def add_product_code_stages(df):

    # 새로운 파생변수 초기화
    df['product_code_stage1'] = 'Other'
    df['product_code_stage2'] = 'Other'
    df['product_code_stage3'] = 'Other'

    # Stage 1 라벨링 적용
    for key, value in Stage1_product_mapping.items():
        stage2_value, stage1_value = key
        df.loc[(df['Stage1 Circle2 Distance Speed Collect Result_Dam'].astype(str) == stage2_value) & 
               (df['Stage1 Circle1 Distance Speed Collect Result_Dam'].astype(str) == stage1_value), 
               'product_code_stage1'] = value

    # Stage 2 라벨링 적용
    for key, value in Stage2_product_mapping.items():
        stage2_value, stage1_value = key
        df.loc[(df['Stage2 Circle2 Distance Speed Collect Result_Dam'].astype(str) == stage2_value) & 
               (df['Stage2 Circle1 Distance Speed Collect Result_Dam'].astype(str) == stage1_value), 
               'product_code_stage2'] = value

    # Stage 3 라벨링 적용
    for key, value in Stage3_product_mapping.items():
        stage2_value, stage1_value = key
        df.loc[(df['Stage3 Circle2 Distance Speed Collect Result_Dam'].astype(str) == stage2_value) & 
               (df['Stage3 Circle1 Distance Speed Collect Result_Dam'].astype(str) == stage1_value), 
               'product_code_stage3'] = value

    # 'product_code_stage' 열을 범주형으로 변환
    df['product_code_stage1'] = df['product_code_stage1'].astype('category')
    df['product_code_stage2'] = df['product_code_stage2'].astype('category')
    df['product_code_stage3'] = df['product_code_stage3'].astype('category')

    return df


def update_code_column(df):

    # 새로운 파생변수 'code' 초기화
    df['code'] = 'Other'

    # 매핑 딕셔너리를 기반으로 'code' 변수 업데이트
    for key, value in code_mapping.items():
        stage1_num, stage2_num, stage3_num = key
        # 포맷팅된 문자열을 사용하여 조건 생성
        stage1_value = f'Stage1_{stage1_num}'
        stage2_value = f'Stage2_{stage2_num}'
        stage3_value = f'Stage3_{stage3_num}'

        df.loc[
            (df['product_code_stage1'] == stage1_value) & 
            (df['product_code_stage2'] == stage2_value) & 
            (df['product_code_stage3'] == stage3_value), 
            'code'
        ] = value

    return df


def cal_fill1_total_dist(data):
    df = data.copy()
    df_selected = pd.DataFrame()
    
    df_selected['Distance (Stage 1)_dam'] = df['DISCHARGED SPEED OF RESIN Collect Result_Dam'] * df['DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam'] * 1000000
    df_selected['Distance (Stage 2)_dam'] = df['DISCHARGED SPEED OF RESIN Collect Result_Dam'] * df['DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam'] * 1000000
    df_selected['Distance (Stage 3)_dam'] = df['DISCHARGED SPEED OF RESIN Collect Result_Dam'] * df['DISCHARGED TIME OF RESIN(Stage3) Collect Result_Dam'] * 1000000

    # 유속 계산
    df_selected['Flow Rate (Stage 1)_dam'] = df['Dispense Volume(Stage1) Collect Result_Dam'] / df['DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam'] * 1000000
    df_selected['Flow Rate (Stage 2)_dam'] = df['Dispense Volume(Stage2) Collect Result_Dam'] / df['DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam'] * 1000000
    df_selected['Flow Rate (Stage 3)_dam'] = df['Dispense Volume(Stage3) Collect Result_Dam'] / df['DISCHARGED TIME OF RESIN(Stage3) Collect Result_Dam'] * 1000000

    # 총 이동 거리 계산
    df_selected['Total Distance Traveled_dam'] = df_selected['Distance (Stage 1)_dam'] + df_selected['Distance (Stage 2)_dam'] + df_selected['Distance (Stage 3)_dam']
    
    # 이동 거리 계산, 
    df_selected['Distance (Stage 1)_fill1'] = df['DISCHARGED SPEED OF RESIN Collect Result_Fill1'] * df['DISCHARGED TIME OF RESIN(Stage1) Collect Result_Fill1'] * 1000000
    df_selected['Distance (Stage 2)_fill1'] = df['DISCHARGED SPEED OF RESIN Collect Result_Fill1'] * df['DISCHARGED TIME OF RESIN(Stage2) Collect Result_Fill1'] * 1000000
    df_selected['Distance (Stage 3)_fill1'] = df['DISCHARGED SPEED OF RESIN Collect Result_Fill1'] * df['DISCHARGED TIME OF RESIN(Stage3) Collect Result_Fill1'] * 1000000

    # 유속 계산, 
    df_selected['Flow Rate (Stage 1)_fill1'] = df['Dispense Volume(Stage1) Collect Result_Fill1'] / df['DISCHARGED TIME OF RESIN(Stage1) Collect Result_Fill1'] * 1000000
    df_selected['Flow Rate (Stage 2)_fill1'] = df['Dispense Volume(Stage2) Collect Result_Fill1'] / df['DISCHARGED TIME OF RESIN(Stage2) Collect Result_Fill1'] * 1000000
    df_selected['Flow Rate (Stage 3)_fill1'] = df['Dispense Volume(Stage3) Collect Result_Fill1'] / df['DISCHARGED TIME OF RESIN(Stage3) Collect Result_Fill1'] * 1000000

    # 총 이동 거리 계산
    df_selected['Total Distance Traveled_fill1'] = df_selected['Distance (Stage 1)_fill1'] + df_selected['Distance (Stage 2)_fill1'] + df_selected['Distance (Stage 3)_fill1']
    

    return df_selected.fillna(0)

# AutoClave_파생 변수 계산 
def derived_autoclave(data):
    df = data.copy()
    df_derived = pd.DataFrame()

    # 압력-시간 곱, 결과에 1000을 곱함
    df_derived['Pressure-Time Product (Stage 1)'] = (
        df['1st Pressure Collect Result_AutoClave'] * 
        df['1st Pressure 1st Pressure Unit Time_AutoClave'] * 1000
    )
    df_derived['Pressure-Time Product (Stage 2)'] = (
        df['2nd Pressure Collect Result_AutoClave'] * 
        df['2nd Pressure Unit Time_AutoClave'] * 1000
    )
    df_derived['Pressure-Time Product (Stage 3)'] = (
        df['3rd Pressure Collect Result_AutoClave'] * 
        df['3rd Pressure Unit Time_AutoClave'] * 1000
    )

    # 각 이전 단계 압력 차이, 결과에 1000을 곱함
    df_derived['Pressure Change (Stage 2)'] = (
        df['2nd Pressure Collect Result_AutoClave'] - df['1st Pressure Collect Result_AutoClave']
    ) * 1000
    df_derived['Pressure Change (Stage 3)'] = (
        df['3rd Pressure Collect Result_AutoClave'] - df['2nd Pressure Collect Result_AutoClave']
    ) * 1000

    return df_derived


In [7]:
sum_columns = [
    [
    'CURE START POSITION X Collect Result_Dam',
    'CURE START POSITION Z Collect Result_Dam',
    'CURE START POSITION Θ Collect Result_Dam',
    ],
    [
    'CURE STANDBY POSITION X Collect Result_Dam',
    'CURE STANDBY POSITION Z Collect Result_Dam',
    'CURE STANDBY POSITION Θ Collect Result_Dam',
    ],
    [
    'HEAD Standby Position X Collect Result_Dam',
    'HEAD Standby Position Y Collect Result_Dam',
    'HEAD Standby Position Z Collect Result_Dam',
    ],
    [
    'Head Clean Position X Collect Result_Dam',
    'Head Clean Position Y Collect Result_Dam',
    'Head Clean Position Z Collect Result_Dam',
    ],
    [
    'Head Purge Position X Collect Result_Dam',
    'Head Purge Position Y Collect Result_Dam',
    'Head Purge Position Z Collect Result_Dam',
    ],
    [
    'Head Zero Position X Collect Result_Dam',
    'Head Zero Position Y Collect Result_Dam',
    'Head Zero Position Z Collect Result_Dam',
    ],
    [
    'HEAD Standby Position X Collect Result_Fill1',
    'HEAD Standby Position Y Collect Result_Fill1',
    'HEAD Standby Position Z Collect Result_Fill1',
    ],
    [
    'Head Clean Position X Collect Result_Fill1',
    'Head Clean Position Y Collect Result_Fill1',
    'Head Clean Position Z Collect Result_Fill1',
    ],
    [
    'Head Purge Position X Collect Result_Fill1',
    'Head Purge Position Y Collect Result_Fill1',
    'Head Purge Position Z Collect Result_Fill1',
    ],

    [
    'CURE END POSITION X Collect Result_Fill2',
    'CURE END POSITION Z Collect Result_Fill2',
    'CURE END POSITION Θ Collect Result_Fill2',
    ],
    [
    'CURE STANDBY POSITION X Collect Result_Fill2',
    'CURE STANDBY POSITION Z Collect Result_Fill2',
    'CURE STANDBY POSITION Θ Collect Result_Fill2',
    ],
    [
    'CURE START POSITION X Collect Result_Fill2',
    'CURE START POSITION Z Collect Result_Fill2',
    'CURE START POSITION Θ Collect Result_Fill2',
    ],
    [
    'HEAD Standby Position X Collect Result_Fill2',
    'HEAD Standby Position Y Collect Result_Fill2',
    'HEAD Standby Position Z Collect Result_Fill2',
    ],
    [
    'Head Clean Position X Collect Result_Fill2',
    'Head Clean Position Y Collect Result_Fill2',
    'Head Clean Position Z Collect Result_Fill2',
    ],
    [
    'Head Purge Position X Collect Result_Fill2',
    'Head Purge Position Y Collect Result_Fill2',
    'Head Purge Position Z Collect Result_Fill2',
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
        'HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Dam',
        'HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Dam',
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Dam',
        'HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Dam',
        'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Dam',
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Dam',
        'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Dam',
        'HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Dam',        
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1',
        'HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill1',
        'HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill1',
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Fill1',
        'HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Fill1',
        'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Fill1',
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill1',
        'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill1',
        'HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Fill1',
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2',
        'HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill2',
        'HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill2',
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Fill2',
        'HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Fill2',
        'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Fill2',
    ],
    [
        'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill2',
        'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill2',
        'HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Fill2',
    ],
]

def sum_positions(data, sum_cols, drop=True):
    df = data.copy()
    for i, col in enumerate(sum_cols):
        try:
            sum_col = col[0] + "_sum"
            df[sum_col] = df[col].sum(axis=1).astype(int)
    #         print(sum_col, df[sum_col].unique(), "\n")
            if drop:
                df = df.drop(col, axis=1)
        except:
            print(col)
    
    return df


In [12]:
drop_col_list = ['DISCHARGED TIME OF RESIN(Stage2) Collect Result_Fill1',
  '1st Pressure 1st Pressure Unit Time_AutoClave',
  'CURE START POSITION Z Collect Result_Fill2',
  'CURE END POSITION Θ Collect Result_Dam',
  'month_Collect Date_AutoClave',
  'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1',
  'DISCHARGED TIME OF RESIN(Stage3) Collect Result_Fill1',
  'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill1',
  'HEAD Standby Position Z Collect Result_Dam',
  'DISCHARGED SPEED OF RESIN Collect Result_Fill1',
  'Machine Tact time Collect Result_Dam',
  'Receip No Collect Result_Dam',
  'HEAD Standby Position X Collect Result_Dam',
  'Dispense Volume(Stage1) Collect Result_Dam',
  'Chamber Temp. Judge Value_AutoClave',
  'CURE END POSITION Z Unit Time_Dam',
  'Wip Line_Dam',
  'Process Desc._Dam',
  'Insp. Seq No._Dam',
  'Insp Judge Code_Dam',
  'CURE END POSITION X Unit Time_Dam',
  'CURE END POSITION X Judge Value_Dam',
  'CURE END POSITION Z Judge Value_Dam',
  'CURE END POSITION Θ Unit Time_Dam',
  'CURE END POSITION Θ Judge Value_Dam',
  'CURE SPEED Unit Time_Dam',
  'CURE SPEED Judge Value_Dam',
  'CURE STANDBY POSITION X Collect Result_Dam',
  'CURE STANDBY POSITION X Unit Time_Dam',
  'CURE STANDBY POSITION X Judge Value_Dam',
  'CURE STANDBY POSITION Z Collect Result_Dam',
  'CURE STANDBY POSITION Z Unit Time_Dam',
  'CURE STANDBY POSITION Z Judge Value_Dam',
  'CURE STANDBY POSITION Θ Collect Result_Dam',
  'CURE STANDBY POSITION Θ Unit Time_Dam',
  'CURE STANDBY POSITION Θ Judge Value_Dam',
  'CURE START POSITION X Unit Time_Dam',
  'CURE START POSITION X Judge Value_Dam',
  'CURE START POSITION Z Collect Result_Dam',
  'CURE START POSITION Z Unit Time_Dam',
  'CURE START POSITION Z Judge Value_Dam',
  'CURE START POSITION Θ Unit Time_Dam',
  'CURE START POSITION Θ Judge Value_Dam',
  'DISCHARGED SPEED OF RESIN Unit Time_Dam',
  'DISCHARGED SPEED OF RESIN Judge Value_Dam',
  'DISCHARGED TIME OF RESIN(Stage1) Unit Time_Dam',
  'DISCHARGED TIME OF RESIN(Stage1) Judge Value_Dam',
  'DISCHARGED TIME OF RESIN(Stage2) Unit Time_Dam',
  'DISCHARGED TIME OF RESIN(Stage2) Judge Value_Dam',
  'DISCHARGED TIME OF RESIN(Stage3) Unit Time_Dam',
  'DISCHARGED TIME OF RESIN(Stage3) Judge Value_Dam',
  'Dispense Volume(Stage1) Unit Time_Dam',
  'Dispense Volume(Stage1) Judge Value_Dam',
  'Dispense Volume(Stage2) Unit Time_Dam',
  'Dispense Volume(Stage2) Judge Value_Dam',
  'Dispense Volume(Stage3) Unit Time_Dam',
  'Dispense Volume(Stage3) Judge Value_Dam',
  'HEAD NORMAL COORDINATE X AXIS(Stage1) Unit Time_Dam',
  'HEAD NORMAL COORDINATE X AXIS(Stage2) Unit Time_Dam',
  'HEAD NORMAL COORDINATE X AXIS(Stage2) Judge Value_Dam',
  'HEAD NORMAL COORDINATE X AXIS(Stage3) Unit Time_Dam',
  'HEAD NORMAL COORDINATE X AXIS(Stage3) Judge Value_Dam',
  'HEAD NORMAL COORDINATE Y AXIS(Stage1) Unit Time_Dam',
  'HEAD NORMAL COORDINATE Y AXIS(Stage1) Judge Value_Dam',
  'HEAD NORMAL COORDINATE Y AXIS(Stage2) Unit Time_Dam',
  'HEAD NORMAL COORDINATE Y AXIS(Stage2) Judge Value_Dam',
  'HEAD NORMAL COORDINATE Y AXIS(Stage3) Unit Time_Dam',
  'HEAD NORMAL COORDINATE Y AXIS(Stage3) Judge Value_Dam',
  'HEAD NORMAL COORDINATE Z AXIS(Stage1) Unit Time_Dam',
  'HEAD NORMAL COORDINATE Z AXIS(Stage1) Judge Value_Dam',
  'HEAD NORMAL COORDINATE Z AXIS(Stage2) Unit Time_Dam',
  'HEAD NORMAL COORDINATE Z AXIS(Stage1) Unit Time_Fill2',
  'HEAD NORMAL COORDINATE Z AXIS(Stage1) Judge Value_Fill2',
  'HEAD NORMAL COORDINATE Z AXIS(Stage2) Unit Time_Fill2',
  'HEAD NORMAL COORDINATE Z AXIS(Stage2) Judge Value_Fill2',
  'HEAD NORMAL COORDINATE Z AXIS(Stage3) Unit Time_Fill2',
  'HEAD NORMAL COORDINATE Z AXIS(Stage3) Judge Value_Fill2',
  'HEAD Standby Position X Unit Time_Fill2',
  'HEAD Standby Position X Judge Value_Fill2',
  'HEAD Standby Position Y Collect Result_Fill2',
  'HEAD Standby Position Y Unit Time_Fill2',
  'HEAD Standby Position Y Judge Value_Fill2',
  'HEAD Standby Position Z Unit Time_Fill2',
  'HEAD Standby Position Z Judge Value_Fill2',
  'Head Clean Position X Unit Time_Fill2',
  'Head Clean Position X Judge Value_Fill2',
  'Head Clean Position Y Collect Result_Fill2',
  'Head Clean Position Y Unit Time_Fill2',
  'Head Clean Position Y Judge Value_Fill2',
  'Head Clean Position Z Unit Time_Fill2',
  'Head Clean Position Z Judge Value_Fill2',
  'Head Purge Position X Collect Result_Fill2',
  'year_Collect Date_Dam',
  'THICKNESS 1 Collect Result_Dam',
  'DISCHARGED TIME OF RESIN(Stage3) Collect Result_Dam',
  '1st Pressure Collect Result_AutoClave',
  'DISCHARGED TIME OF RESIN(Stage1) Collect Result_Fill1',
  'Dispense Volume(Stage3) Collect Result_Dam',
  'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1',
  'HEAD Standby Position Y Collect Result_Fill1',
  'THICKNESS 2 Collect Result_Dam',
  'CURE SPEED Collect Result_Dam',
  'GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave',
  'year_Collect Date_Fill1',
  'HEAD Standby Position X Collect Result_Fill1',
  'Production Qty Collect Result_Dam',
  'CURE END POSITION X Collect Result_Dam',
  'Head Purge Position X Unit Time_Fill2',
  'Head Purge Position X Judge Value_Fill2',
  'Head Purge Position Y Collect Result_Fill2',
  'Head Purge Position Y Unit Time_Fill2',
  'Head Purge Position Y Judge Value_Fill2',
  'Head Purge Position Z Collect Result_Fill2',
  'Head Purge Position Z Unit Time_Fill2',
  'Head Purge Position Z Judge Value_Fill2',
  'CURE START POSITION Θ Collect Result_Dam',
  'Machine Tact time Unit Time_Fill2',
  'Machine Tact time Judge Value_Fill2',
  'PalletID Unit Time_Fill2',
  'PalletID Judge Value_Fill2',
  'Production Qty Unit Time_Fill2',
  'Production Qty Judge Value_Fill2',
  'Receip No Unit Time_Fill2',
  'Receip No Judge Value_Fill2',
  'WorkMode Unit Time_Fill2',
  'WorkMode Judge Value_Fill2',
  'year_Collect Date_AutoClave',
  'code',
  'HEAD NORMAL COORDINATE Z AXIS(Stage2) Judge Value_Dam',
  'HEAD NORMAL COORDINATE Z AXIS(Stage3) Unit Time_Dam',
  'HEAD NORMAL COORDINATE Z AXIS(Stage3) Judge Value_Dam',
  'HEAD Standby Position X Unit Time_Dam',
  'HEAD Standby Position X Judge Value_Dam',
  'HEAD Standby Position Y Unit Time_Dam',
  'HEAD Standby Position Y Judge Value_Dam',
  'HEAD Standby Position Z Unit Time_Dam',
  'HEAD Standby Position Z Judge Value_Dam',
  'Head Clean Position X Unit Time_Dam',
  'Head Clean Position X Judge Value_Dam',
  'Head Clean Position Y Unit Time_Dam',
  'Head Clean Position Y Judge Value_Dam',
  'Head Clean Position Z Unit Time_Dam',
  'Head Clean Position Z Judge Value_Dam',
  'Head Purge Position X Unit Time_Dam',
  'Head Purge Position X Judge Value_Dam',
  'Head Purge Position Y Unit Time_Dam',
  'Head Purge Position Y Judge Value_Dam',
  'Head Purge Position Z Unit Time_Dam',
  'Head Purge Position Z Judge Value_Dam',
  'Head Zero Position X Collect Result_Dam',
  'Head Zero Position X Unit Time_Dam',
  'Head Zero Position X Judge Value_Dam',
  'Head Zero Position Y Unit Time_Dam',
  'Head Zero Position Y Judge Value_Dam',
  'Head Zero Position Z Unit Time_Dam',
  'Head Zero Position Z Judge Value_Dam',
  'Machine Tact time Unit Time_Dam',
  'Machine Tact time Judge Value_Dam',
  'PalletID Unit Time_Dam',
  'PalletID Judge Value_Dam',
  'Production Qty Unit Time_Dam',
  'Production Qty Judge Value_Dam',
  'Receip No Unit Time_Dam',
  'Receip No Judge Value_Dam',
  'Stage1 Circle1 Distance Speed Unit Time_Dam',
  'Stage1 Circle1 Distance Speed Judge Value_Dam',
  'Dispense Volume(Stage2) Collect Result_Fill1',
  'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2',
  'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam',
  'Machine Tact time Collect Result_Fill1',
  'PalletID Collect Result_Fill1',
  'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Fill1',
  'Head Clean Position X Collect Result_Fill1',
  'Head Clean Position X Collect Result_Dam',
  'HEAD Standby Position Z Collect Result_Fill2',
  'HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Fill2',
  'Equipment_Fill1',
  'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2',
  'HEAD Standby Position Y Collect Result_Dam',
  'THICKNESS 3 Collect Result_Dam',
  'Receip No Collect Result_Fill2',
  '2nd Pressure Unit Time_AutoClave',
  'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Fill2',
  'Stage1 Circle2 Distance Speed Unit Time_Dam',
  'Stage1 Circle2 Distance Speed Judge Value_Dam',
  'Stage1 Circle3 Distance Speed Unit Time_Dam',
  'Stage1 Circle3 Distance Speed Judge Value_Dam',
  'Stage1 Circle4 Distance Speed Unit Time_Dam',
  'Stage1 Circle4 Distance Speed Judge Value_Dam',
  'Stage1 Line1 Distance Speed Unit Time_Dam',
  'Stage1 Line1 Distance Speed Judge Value_Dam',
  'Stage1 Line2 Distance Speed Unit Time_Dam',
  'Stage1 Line2 Distance Speed Judge Value_Dam',
  'Stage1 Line3 Distance Speed Unit Time_Dam',
  'Stage1 Line3 Distance Speed Judge Value_Dam',
  'Stage1 Line4 Distance Speed Unit Time_Dam',
  'Stage1 Line4 Distance Speed Judge Value_Dam',
  'Stage2 Circle1 Distance Speed Unit Time_Dam',
  'Stage2 Circle1 Distance Speed Judge Value_Dam',
  'Stage2 Circle2 Distance Speed Unit Time_Dam',
  'Stage2 Circle2 Distance Speed Judge Value_Dam',
  'Stage2 Circle3 Distance Speed Unit Time_Dam',
  'Stage2 Circle3 Distance Speed Judge Value_Dam',
  'Stage2 Circle4 Distance Speed Unit Time_Dam',
  'Stage2 Circle4 Distance Speed Judge Value_Dam',
  'Stage2 Line1 Distance Speed Unit Time_Dam',
  'Stage2 Line1 Distance Speed Judge Value_Dam',
  'Stage2 Line2 Distance Speed Unit Time_Dam',
  'Insp. Seq No._Fill1',
  'Stage2 Line2 Distance Speed Judge Value_Dam',
  'Stage2 Line3 Distance Speed Unit Time_Dam',
  'Stage2 Line3 Distance Speed Judge Value_Dam',
  'Stage2 Line4 Distance Speed Unit Time_Dam',
  'Stage2 Line4 Distance Speed Judge Value_Dam',
  'Stage3 Circle1 Distance Speed Unit Time_Dam',
  'Stage3 Circle1 Distance Speed Judge Value_Dam',
  'Stage3 Circle2 Distance Speed Unit Time_Dam',
  'Stage3 Circle2 Distance Speed Judge Value_Dam',
  'Stage3 Circle3 Distance Speed Unit Time_Dam',
  'Stage3 Circle3 Distance Speed Judge Value_Dam',
  'Stage3 Circle4 Distance Speed Unit Time_Dam',
  'Stage3 Circle4 Distance Speed Judge Value_Dam',
  'Stage3 Line1 Distance Speed Unit Time_Dam',
  'Stage3 Line1 Distance Speed Judge Value_Dam',
  'Stage3 Line2 Distance Speed Unit Time_Dam',
  'Dispense Volume(Stage3) Collect Result_Fill1',
  'DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam',
  'Machine Tact time Collect Result_Fill2',
  'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill1',
  'Receip No Collect Result_Fill1',
  'DISCHARGED SPEED OF RESIN Collect Result_Dam',
  'Production Qty Collect Result_Fill1',
  'HEAD Standby Position Z Collect Result_Fill1',
  'Stage3 Line2 Distance Speed Judge Value_Dam',
  'Stage3 Line3 Distance Speed Unit Time_Dam',
  'Stage3 Line3 Distance Speed Judge Value_Dam',
  'Stage3 Line4 Distance Speed Unit Time_Dam',
  'Stage3 Line4 Distance Speed Judge Value_Dam',
  'THICKNESS 1 Unit Time_Dam',
  'THICKNESS 1 Judge Value_Dam',
  'THICKNESS 2 Unit Time_Dam',
  'THICKNESS 2 Judge Value_Dam',
  'THICKNESS 3 Unit Time_Dam',
  'THICKNESS 3 Judge Value_Dam',
  'WorkMode Unit Time_Dam',
  'WorkMode Judge Value_Dam',
  'Wip Line_AutoClave',
  'Process Desc._AutoClave',
  'Equipment_AutoClave',
  'Insp. Seq No._AutoClave',
  'Insp Judge Code_AutoClave',
  '1st Pressure Judge Value_AutoClave',
  '2nd Pressure Judge Value_AutoClave',
  '3rd Pressure Judge Value_AutoClave',
  'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave',
  'GMES_ORIGIN_INSP_JUDGE_CODE Unit Time_AutoClave',
  'Wip Line_Fill1',
  'Process Desc._Fill1',
  'Insp Judge Code_Fill1',
  'DISCHARGED SPEED OF RESIN Unit Time_Fill1',
  'DISCHARGED SPEED OF RESIN Judge Value_Fill1',
  'DISCHARGED TIME OF RESIN(Stage1) Unit Time_Fill1',
  'DISCHARGED TIME OF RESIN(Stage1) Judge Value_Fill1',
  'DISCHARGED TIME OF RESIN(Stage2) Unit Time_Fill1',
  'DISCHARGED TIME OF RESIN(Stage2) Judge Value_Fill1',
  'DISCHARGED TIME OF RESIN(Stage3) Unit Time_Fill1',
  'DISCHARGED TIME OF RESIN(Stage3) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE Y AXIS(Stage3) Unit Time_Fill2',
  'HEAD NORMAL COORDINATE Y AXIS(Stage3) Judge Value_Fill2',
  'Dispense Volume(Stage1) Unit Time_Fill1',
  'Dispense Volume(Stage1) Judge Value_Fill1',
  'Dispense Volume(Stage2) Unit Time_Fill1',
  'Dispense Volume(Stage2) Judge Value_Fill1',
  'Dispense Volume(Stage3) Unit Time_Fill1',
  'Dispense Volume(Stage3) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE X AXIS(Stage1) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE X AXIS(Stage2) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE X AXIS(Stage2) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE X AXIS(Stage3) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE X AXIS(Stage3) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE Y AXIS(Stage1) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE Y AXIS(Stage1) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE Y AXIS(Stage2) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE Y AXIS(Stage2) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE Y AXIS(Stage3) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE Y AXIS(Stage3) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE Z AXIS(Stage1) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE Z AXIS(Stage1) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE Z AXIS(Stage2) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE Z AXIS(Stage2) Judge Value_Fill1',
  'HEAD NORMAL COORDINATE Z AXIS(Stage3) Unit Time_Fill1',
  'HEAD NORMAL COORDINATE Z AXIS(Stage3) Judge Value_Fill1',
  'HEAD Standby Position X Unit Time_Fill1',
  'HEAD Standby Position X Judge Value_Fill1',
  'HEAD Standby Position Y Unit Time_Fill1',
  'HEAD Standby Position Y Judge Value_Fill1',
  'HEAD Standby Position Z Unit Time_Fill1',
  'HEAD Standby Position Z Judge Value_Fill1',
  'Head Clean Position X Unit Time_Fill1',
  'Head Clean Position X Judge Value_Fill1',
  'Head Clean Position Y Unit Time_Fill1',
  'Head Clean Position Y Judge Value_Fill1',
  'Head Clean Position Z Unit Time_Fill1',
  'Head Clean Position Z Judge Value_Fill1',
  'Head Purge Position X Unit Time_Fill1',
  'Head Purge Position X Judge Value_Fill1',
  'Head Purge Position Y Unit Time_Fill1',
  'Head Purge Position Y Judge Value_Fill1',
  'Head Purge Position Z Unit Time_Fill1',
  'Head Purge Position Z Judge Value_Fill1']

In [11]:
def base_processing(data):
    df = data.copy()
    columns_to_replace = [
        'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam',
        'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1',
        'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2',
    ]

    df[columns_to_replace] = df[columns_to_replace].replace('OK', 0)
    df[columns_to_replace] = df[columns_to_replace].fillna(0).astype(float)

    df = df.fillna(0)
#     df = df.drop(['Workorder_Dam'], axis=1)
    
    # 시간 컬럼 분리
    date_cols = ['Collect Date_Dam', 'Collect Date_Fill1', 'Collect Date_Fill2', 'Collect Date_AutoClave']
    for date_col in date_cols:
        df = split_date(df, date_col, drop=True)
        
    # 제품군 코드
    df = add_product_code_stages(df)
    df = update_code_column(df)
    df = df.drop(columns=Circles_lines_drop)
    
#     df = cure(df)
#     df = head_position(df)
    
    # 좌표 통합
    df = sum_positions(df, sum_columns)
    

#     df_selected = cal_fill1_total_dist(df)
#     df_derived = derived_autoclave(df)
#     df = pd.concat([df, df_derived, df_selected], axis=1)

    
    for col in df.select_dtypes(include=['number']).columns:
        df[col] = (df[col]*1000).astype(int) 
    for col in df.columns:
        df[col] = df[col].astype('category') 

    return df

ROOT_DIR = "data"
RANDOM_STATE = 999

train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))
hand_data = pd.read_excel(os.path.join(ROOT_DIR, "hand_data.xlsx"))

train_data, test_data = process_and_merge_data(train_data, test_data, hand_data)

# # 핸드 라벨링 후처리를 위한 인덱스 남겨두기
# diff = create_diff_column(test_data, 'Collect Date_Fill1', 'Collect Date_Fill2')
# abnormal_diff_index = diff[~((diff == '0 days 00:00:00') | (diff == '0 days 00:10:00'))].index

train_data = base_processing(train_data)
test_data = base_processing(test_data)

['HEAD Standby Position X Collect Result_Dam', 'HEAD Standby Position Y Collect Result_Dam', 'HEAD Standby Position Z Collect Result_Dam']
['Head Clean Position X Collect Result_Dam', 'Head Clean Position Y Collect Result_Dam', 'Head Clean Position Z Collect Result_Dam']
['Head Zero Position X Collect Result_Dam', 'Head Zero Position Y Collect Result_Dam', 'Head Zero Position Z Collect Result_Dam']
['HEAD Standby Position X Collect Result_Fill1', 'HEAD Standby Position Y Collect Result_Fill1', 'HEAD Standby Position Z Collect Result_Fill1']
['Head Clean Position X Collect Result_Fill1', 'Head Clean Position Y Collect Result_Fill1', 'Head Clean Position Z Collect Result_Fill1']
['HEAD Standby Position X Collect Result_Fill2', 'HEAD Standby Position Y Collect Result_Fill2', 'HEAD Standby Position Z Collect Result_Fill2']
['Head Clean Position X Collect Result_Fill2', 'Head Clean Position Y Collect Result_Fill2', 'Head Clean Position Z Collect Result_Fill2']
['HEAD Standby Position X Coll

In [13]:
len(drop_col_list)

310

In [14]:
drop_col_list
train_data = train_data.drop(columns=[col for col in drop_col_list if col in train_data.columns])
train_data.shape

(40506, 157)

In [15]:
x_train = train_data.drop('target', axis=1)
y_train = train_data["target"]

features = list(x_train.columns)
x_train.shape

x_test = test_data[features]
x_test.shape

(17361, 156)

In [24]:
from sklearn.model_selection import StratifiedKFold

def generate_datasets(X, y, n_splits_list, random_state_list, sampling=False):
    datasets = []
    
    for n_splits, random_state in zip(n_splits_list, random_state_list):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        
        for train_index, test_index in skf.split(X, y):
            X_tr, X_test = X.iloc[train_index], X.iloc[test_index]
            y_tr, y_test = y.iloc[train_index], y.iloc[test_index]
            
            if sampling:
                rus = RandomUnderSampler(sampling_strategy=1, random_state=random_state)
                X_tr, y_tr = rus.fit_resample(X_tr, y_tr)
            
            datasets.append((X_tr, y_tr, X_test, y_test))
    
    return datasets

n_splits_list = [10]
random_state_list = [
    RANDOM_STATE
]
datasets = generate_datasets(x_train, y_train, n_splits_list, random_state_list, sampling=False)
len(datasets)

10

## 3. 모델 학습


### 모델 학습


In [26]:
from sklearn.linear_model import LogisticRegression  # Meta-model
import numpy as np

def train_stacking_ensemble_model(datasets, params, x_test):
    level1_train_predictions = []
    level1_test_predictions = []
    y_true_val = []

    # Stage 1: Train base models and collect validation predictions
    for param in params:
        print("학습 파라미터: ", param)
        fold_train_predictions = []
        fold_test_predictions = []
        
        for i, (X_train, y_tr, X_val, y_val) in enumerate(datasets):
            print(f"{i+1}번째 데이터셋")
            model = CatBoostClassifier(
                **param,
                random_seed=RANDOM_STATE, 
                iterations=1000,
                od_wait=50,
                verbose=100,
                use_best_model=True,
                task_type='GPU'
            )

            # CatBoost 학습데이터
            train_pool = Pool(
                X_train,
                label=y_tr,
                cat_features=features
            )
            # CatBoost 검증데이터 
            eval_pool = Pool(
                X_val,
                label=y_val,
                cat_features=features
            )

            model.fit(train_pool, eval_set=eval_pool)
            val_pred = model.predict_proba(X_val)[:, 1]  # Use probabilities for stacking
            print(f"Fold {i+1} - Validation F1: {model.get_best_score()}\n\n")

            fold_train_predictions.append(val_pred)  # Collect validation predictions
            y_true_val.append(y_val)  # Collect true labels
            
            # Predictions on test data
            test_pool = Pool(data=x_test, cat_features=features)
            y_test_prob = model.predict_proba(test_pool)[:, 1]  # Use probabilities for stacking
            fold_test_predictions.append(y_test_prob)

        # Collect the predictions from all folds for this base model
        level1_train_predictions.append(np.concatenate(fold_train_predictions, axis=0))  # Stacking train set
        level1_test_predictions.append(np.mean(fold_test_predictions, axis=0))  # Stacking test set

        print('-'*20, '\n')
        
    return level1_train_predictions, level1_test_predictions

n_splits_list = [10]
random_state_list = [
    RANDOM_STATE
]
datasets = generate_datasets(x_train, y_train, n_splits_list, random_state_list, sampling=False)

params = [
    {'learning_rate': 0.05,
     'eval_metric': 'F1',
     'depth': 5,
    },
    {'learning_rate': 0.05,
     'eval_metric': 'F1',
     'depth': 6,
    },
    {'learning_rate': 0.05,
     'eval_metric': 'F1',
     'depth': 7,
    },
    {'learning_rate': 0.05,
     'eval_metric': 'F1',
     'depth': 8,
    },
]

# Call the stacking ensemble model
level1_train_predictions, level1_test_predictions = train_stacking_ensemble_model(datasets, params, x_test)

학습 파라미터:  {'learning_rate': 0.05, 'eval_metric': 'F1', 'depth': 5}
1번째 데이터셋
0:	learn: 0.9704099	test: 0.9699898	best: 0.9699898 (0)	total: 61.8ms	remaining: 1m 1s
100:	learn: 0.9722466	test: 0.9729730	best: 0.9729730 (98)	total: 6.67s	remaining: 59.4s
200:	learn: 0.9730572	test: 0.9738420	best: 0.9738420 (156)	total: 13.2s	remaining: 52.4s
bestTest = 0.9738420314
bestIteration = 156
Shrink model to first 157 iterations.
Fold 1 - Validation F1: {'learn': {'Logloss': 0.1933804840728295, 'F1': 0.9730572444973566}, 'validation': {'Logloss': 0.18896157427853463, 'F1': 0.9738420313895624}}


2번째 데이터셋
0:	learn: 0.9702889	test: 0.9702366	best: 0.9702366 (0)	total: 57.4ms	remaining: 57.4s
100:	learn: 0.9719478	test: 0.9719602	best: 0.9722222 (91)	total: 6.58s	remaining: 58.6s
200:	learn: 0.9731967	test: 0.9727041	best: 0.9727041 (187)	total: 13.2s	remaining: 52.3s
bestTest = 0.9727040816
bestIteration = 187
Shrink model to first 188 iterations.
Fold 2 - Validation F1: {'learn': {'Logloss': 0.19

8번째 데이터셋
0:	learn: 0.9703841	test: 0.9703525	best: 0.9703525 (0)	total: 75.6ms	remaining: 1m 15s
100:	learn: 0.9725826	test: 0.9720912	best: 0.9720912 (97)	total: 8.51s	remaining: 1m 15s
200:	learn: 0.9734205	test: 0.9727110	best: 0.9727110 (164)	total: 16.8s	remaining: 1m 6s
bestTest = 0.9729591837
bestIteration = 225
Shrink model to first 226 iterations.
Fold 8 - Validation F1: {'learn': {'Logloss': 0.18865816787274386, 'F1': 0.9735446734907988}, 'validation': {'Logloss': 0.1975114685812114, 'F1': 0.9729591836734695}}


9번째 데이터셋
0:	learn: 0.9703986	test: 0.9702214	best: 0.9702214 (0)	total: 74.3ms	remaining: 1m 14s
100:	learn: 0.9720878	test: 0.9719745	best: 0.9719745 (97)	total: 8.58s	remaining: 1m 16s
200:	learn: 0.9731859	test: 0.9727110	best: 0.9727110 (163)	total: 16.9s	remaining: 1m 7s
bestTest = 0.9730833014
bestIteration = 219
Shrink model to first 220 iterations.
Fold 9 - Validation F1: {'learn': {'Logloss': 0.19010155489237027, 'F1': 0.9734059142071391}, 'validation': {'Log

0:	learn: 0.9706261	test: 0.9707379	best: 0.9707379 (0)	total: 167ms	remaining: 2m 47s
100:	learn: 0.9725389	test: 0.9719602	best: 0.9719602 (100)	total: 18s	remaining: 2m 40s
bestTest = 0.9725800281
bestIteration = 134
Shrink model to first 135 iterations.
Fold 4 - Validation F1: {'learn': {'Logloss': 0.19020745289569332, 'F1': 0.9734894666704472}, 'validation': {'Logloss': 0.1968741773587926, 'F1': 0.9725800280576458}}


5번째 데이터셋
0:	learn: 0.9703687	test: 0.9706144	best: 0.9706144 (0)	total: 168ms	remaining: 2m 47s
bestTest = 0.9713631157
bestIteration = 27
Shrink model to first 28 iterations.
Fold 5 - Validation F1: {'learn': {'Logloss': 0.19958182337385133, 'F1': 0.971785148290695}, 'validation': {'Logloss': 0.19994354813047704, 'F1': 0.9713631156930126}}


6번째 데이터셋
0:	learn: 0.9702296	test: 0.9704759	best: 0.9704759 (0)	total: 84.2ms	remaining: 1m 24s
100:	learn: 0.9719891	test: 0.9722293	best: 0.9722293 (92)	total: 17.9s	remaining: 2m 39s
200:	learn: 0.9733507	test: 0.9735801	bes

## 스태킹 앙상블 결과 저장하기

### 후처리 인덱스

In [27]:
# 핸드 라벨링 후처리를 위한 인덱스 남겨두기
temp = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))

diff = create_diff_column(temp, 'Collect Date_Fill1', 'Collect Date_Fill2')
abnormal_diff_index = diff[~((diff == '0 days 00:00:00') | (diff == '0 days 00:10:00'))].index
em = add_equipment_match_column(temp)
abnormal_em_index = em[em['equipment_match'] == 0 ].index

In [28]:
y_true_val = []
for i, (X_train, y_tr, X_val, y_val) in enumerate(datasets):
    y_true_val.append(y_val)

In [31]:
# Stage 2: Train meta-model on the level 1 predictions
# Stack the base model predictions for the meta-model
X_meta_train = np.vstack(level1_train_predictions).T
X_meta_test = np.vstack(level1_test_predictions).T
y_meta_train = np.concatenate(y_true_val, axis=0)

# Train a simple logistic regression meta-model
meta_model = LogisticRegression(random_state=RANDOM_STATE)
meta_model.fit(X_meta_train, y_meta_train)

# Final predictions from the meta-model
final_test_prob = meta_model.predict_proba(X_meta_test)[:, 1]
final_test_pred = np.where(final_test_prob < 0.9, 'AbNormal', 'Normal')

print("Final 'AbNormal' predictions in test set:", sum(final_test_pred == 'AbNormal'))

df_sub = pd.read_csv("submission.csv")
df_sub["target"] = final_test_pred

# 후처리 (핸드라벨링)
df_sub.iloc[abnormal_em_index]['target'] = 'Abnormal'
df_sub.iloc[abnormal_diff_index]['target'] = 'Abnormal'

# 제출 파일 저장
df_sub.to_csv("submission.csv", index=False)

Final 'AbNormal' predictions in test set: 614


/tmp/ipykernel_12276/2492258877.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub.iloc[abnormal_em_index]['target'] = 'Abnormal'
/tmp/ipykernel_12276/2492258877.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub.iloc[abnormal_diff_index]['target'] = 'Abnormal'


In [ ]:
0.28320802005012535